# Turbojet Parametric Explorer
### AE 3310 · Section 6 — Interactive Cycle Analysis

Use the sliders below to explore how the flow through a turbojet changes with key design parameters.
Each section has a **question** about what you expect to see — click **▶ Reveal Answer** after you've thought about it.

---


In [ ]:
import sys, os, warnings
warnings.filterwarnings('ignore')
sys.path.insert(0, os.path.dirname(os.path.abspath('.')))

import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from io import BytesIO
import base64
import openmdao.api as om
import pycycle.api as pyc
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

# ── pyCycle model (built once) ─────────────────────────────────────────────────
class Single(pyc.MPCycle):
    def setup(self):
        from simple_turbojet import Turbojet
        self.pyc_add_pnt('DESIGN', Turbojet())
        self.set_input_defaults('DESIGN.Nmech', 8070.0, units='rpm')
        self.set_input_defaults('DESIGN.inlet.MN', 0.60)
        self.set_input_defaults('DESIGN.comp.MN', 0.020)
        self.set_input_defaults('DESIGN.burner.MN', 0.020)
        self.set_input_defaults('DESIGN.turb.MN', 0.4)
        self.pyc_add_cycle_param('burner.dPqP', 0.03)
        self.pyc_add_cycle_param('nozz.Cv', 0.99)
        super().setup()

print("Building pyCycle model (one-time setup)...")
_prob = om.Problem()
_prob.model = Single()
_prob.setup(check=False)
_prob.set_solver_print(level=-1)

# Warm-start initial guesses
_prob['DESIGN.balance.FAR']      = 0.0175506829934
_prob['DESIGN.balance.W']        = 168.453135137
_prob['DESIGN.balance.turb_PR']  = 4.46138725662
_prob['DESIGN.fc.balance.Pt']    = 14.6955113159
_prob['DESIGN.fc.balance.Tt']    = 518.665288153

STATION_VARS   = [('fc.Fl_O','stat:T','stat:P'), ('inlet.Fl_O','tot:T','tot:P'),
                  ('comp.Fl_O','tot:T','tot:P'),  ('burner.Fl_O','tot:T','tot:P'),
                  ('turb.Fl_O','tot:T','tot:P'),  ('nozz.Fl_O','tot:T','tot:P')]
STATION_LABELS = ['Freestream','Inlet','Compressor','Burner','Turbine','Nozzle']
DARK='#0d1117'; CARD='#161b22'; GRID='#21262d'; TEXT='#f0f6fc'; MUTED='#8b949e'

def run_design(T4=2370, PR=13.5, eta_c=0.83, eta_t=0.86):
    _prob.set_val('DESIGN.fc.alt', 0, units='ft')
    _prob.set_val('DESIGN.fc.MN', 0.000001)
    _prob.set_val('DESIGN.balance.Fn_target', 11800.0, units='lbf')
    _prob.set_val('DESIGN.balance.T4_target', T4, units='degR')
    _prob.set_val('DESIGN.comp.PR', PR)
    _prob.set_val('DESIGN.comp.eff', eta_c)
    _prob.set_val('DESIGN.turb.eff', eta_t)
    _prob.run_model()
    Tt = [float(_prob.get_val(f'DESIGN.{fs}:{tv}', units='degR')[0]) for fs,tv,_ in STATION_VARS]
    Pt = [float(_prob.get_val(f'DESIGN.{fs}:{pv}', units='lbf/inch**2')[0]) for fs,_,pv in STATION_VARS]
    Fn   = float(_prob.get_val('DESIGN.perf.Fn',   units='lbf')[0])
    TSFC = float(_prob.get_val('DESIGN.perf.TSFC', units='lbm/(lbf*h)')[0])
    OPR  = float(_prob.get_val('DESIGN.perf.OPR')[0])
    return np.array(Tt), np.array(Pt), Fn, TSFC, OPR

def dark_ax(ax):
    ax.set_facecolor(CARD)
    for s in ax.spines.values(): s.set_color(GRID)
    ax.tick_params(colors=MUTED, labelsize=9)
    ax.xaxis.label.set_color(MUTED); ax.yaxis.label.set_color(MUTED)
    ax.title.set_color(TEXT); ax.grid(True, color=GRID, lw=0.8)

def fig_to_img(fig):
    buf = BytesIO()
    fig.savefig(buf, format='png', dpi=120, bbox_inches='tight', facecolor=DARK)
    plt.close(fig)
    buf.seek(0)
    return 'data:image/png;base64,' + base64.b64encode(buf.read()).decode()

def make_station_fig(Tt, Pt, Tt_ref, Pt_ref, param_name, param_val):
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    fig.patch.set_facecolor(DARK)
    for ax in axes: dark_ax(ax)
    axes[0].plot(STATION_LABELS, Tt_ref, 'o--', color='#30363d', lw=1.5, ms=5, label='Design ref')
    axes[0].plot(STATION_LABELS, Tt, 'o-',  color='#06b6d4', lw=2.5, ms=7, label=f'{param_name}={param_val}')
    axes[0].set_title('Total Temperature (°R)', fontsize=11)
    axes[0].legend(fontsize=8, facecolor=DARK, labelcolor=TEXT, edgecolor=GRID)
    axes[1].plot(STATION_LABELS, Pt_ref, 's--', color='#30363d', lw=1.5, ms=5, label='Design ref')
    axes[1].plot(STATION_LABELS, Pt, 's-',  color='#f59e0b', lw=2.5, ms=7, label=f'{param_name}={param_val}')
    axes[1].set_title('Total Pressure (psi)', fontsize=11)
    axes[1].legend(fontsize=8, facecolor=DARK, labelcolor=TEXT, edgecolor=GRID)
    return fig

def make_ts_fig(Tt, Pt, Tt_ref, Pt_ref, param_name, param_val):
    gamma, cp_btu = 1.4, 0.24; R_btu = cp_btu*(gamma-1)/gamma
    def entropy(T, P):
        s = [0.0]
        for i in range(1, len(T)):
            s.append(s[-1] + cp_btu*np.log(T[i]/T[i-1]) - R_btu*np.log(P[i]/P[i-1]))
        return np.array(s)
    s_ref = entropy(Tt_ref, Pt_ref)
    s_new = entropy(Tt, Pt)
    fig, ax = plt.subplots(figsize=(7, 4)); fig.patch.set_facecolor(DARK); dark_ax(ax)
    ax.plot(s_ref, Tt_ref, 'o--', color='#30363d', lw=1.5, ms=5, label='Design ref')
    ax.plot(s_new, Tt,     'o-',  color='#a78bfa', lw=2.5, ms=7, label=f'{param_name}={param_val}')
    for j, lbl in enumerate(STATION_LABELS):
        ax.annotate(lbl[:4], (s_new[j], Tt[j]), textcoords='offset points', xytext=(4,4), fontsize=8, color='#a78bfa', alpha=0.8)
    ax.set_xlabel('Relative Entropy (BTU/lb·°R)', fontsize=10); ax.set_ylabel('Tt (°R)', fontsize=10)
    ax.set_title('T–s Diagram', fontsize=11)
    ax.legend(fontsize=8, facecolor=DARK, labelcolor=TEXT, edgecolor=GRID)
    return fig

# Run design reference point
print("Running design reference point...")
Tt_ref, Pt_ref, Fn_ref, TSFC_ref, OPR_ref = run_design()
print(f"  Design: Fn={Fn_ref:.1f} lbf  TSFC={TSFC_ref:.5f}  OPR={OPR_ref:.3f}")
print("Ready! Scroll down and use the sliders.")


In [ ]:
def reveal_button(question, answer):
    q_html = widgets.HTML(
        f'''<div style="background:#161b22;border-left:4px solid #f59e0b;
                        padding:12px 16px;border-radius:4px;margin:10px 0;font-family:monospace;color:#f0f6fc;font-size:13px;">
            <b style="color:#f59e0b;">&#x1F914; Think about it:</b><br>{question}
           </div>'''
    )
    btn = widgets.Button(
        description='▶ Reveal Answer',
        style={'button_color': '#1a3a5c'},
        layout=widgets.Layout(width='160px', height='32px')
    )
    ans_out = widgets.Output()
    def on_click(b):
        ans_out.clear_output()
        with ans_out:
            display(widgets.HTML(
                f'''<div style="background:#0d2818;border-left:4px solid #10b981;
                                padding:12px 16px;border-radius:4px;margin:6px 0;
                                font-family:sans-serif;color:#f0f6fc;font-size:13px;">
                    <b style="color:#10b981;">&#x2705; Answer:</b><br>{answer}
                   </div>'''
            ))
    btn.on_click(on_click)
    return widgets.VBox([q_html, btn, ans_out])


---
## 🔥 Section 1 — Turbine Inlet Temperature (T4)

T4 is the temperature at the exit of the combustor — the hottest point in the cycle.
It's limited by turbine blade material limits (~2500–2900°R for modern engines with cooling).


In [ ]:
T4_slider = widgets.FloatSlider(
    value=2370, min=1900, max=2700, step=50,
    description='T4 (°R):', style={'description_width': '80px'},
    layout=widgets.Layout(width='500px'),
    continuous_update=False
)
T4_out  = widgets.Output()
T4_perf = widgets.HTML()

def update_T4(change):
    T4_out.clear_output(wait=True)
    T4 = T4_slider.value
    Tt, Pt, Fn, TSFC, OPR = run_design(T4=T4)
    T4_perf.value = (
        f'<div style="font-family:monospace;color:#f0f6fc;background:#161b22;'
        f'padding:8px 16px;border-radius:6px;display:inline-block;margin:6px 0;">'
        f'Fn = <b style="color:#06b6d4">{Fn:.1f} lbf</b> &nbsp;|&nbsp; '
        f'TSFC = <b style="color:#f59e0b">{TSFC:.5f}</b> &nbsp;|&nbsp; '
        f'OPR = <b style="color:#a78bfa">{OPR:.3f}</b>'
        f'</div>'
    )
    with T4_out:
        fig1 = make_station_fig(Tt, Pt, Tt_ref, Pt_ref, 'T4', f'{T4:.0f}°R')
        display(widgets.HTML(f'<img src="{fig_to_img(fig1)}" style="max-width:100%">'))
        fig2 = make_ts_fig(Tt, Pt, Tt_ref, Pt_ref, 'T4', f'{T4:.0f}°R')
        display(widgets.HTML(f'<img src="{fig_to_img(fig2)}" style="max-width:100%">'))

T4_slider.observe(update_T4, names='value')

q1 = reveal_button(
    "If you increase T4 from 2370°R to 2600°R while holding thrust constant at 11,800 lbf, "
    "what do you expect to happen to TSFC? Why does the station pressure profile barely change?",
    "TSFC <b>increases</b> — you need more fuel to reach a higher T4, but the turbine must extract "
    "more work to maintain shaft balance, so the nozzle exit conditions change only modestly. "
    "Pressure is set by the compressor PR (fixed at 13.5), so the Pt profile stays almost identical — "
    "T4 only controls how much heat the burner adds between the same pressure levels."
)
q2 = reveal_button(
    "Where does the extra energy from raising T4 go if thrust is held constant by the balance?",
    "The balance adjusts <b>mass flow rate W</b> to hit the thrust target — so raising T4 "
    "actually <b>reduces W</b> slightly (hotter gas has higher specific thrust). "
    "The net effect is more heat per unit mass flow, which shows up as higher TSFC."
)

display(widgets.VBox([T4_slider, T4_perf, q1, q2, T4_out]))
update_T4(None)


---
## ⚙️ Section 2 — Compressor Pressure Ratio (PR)

The compressor PR sets how much the air is compressed before entering the combustor.
Higher PR → better thermal efficiency, but compressor exit temperature rises sharply.
Modern turbojets run PR ≈ 10–30.


In [ ]:
PR_slider = widgets.FloatSlider(
    value=13.5, min=5, max=25, step=0.5,
    description='PR:', style={'description_width': '80px'},
    layout=widgets.Layout(width='500px'),
    continuous_update=False
)
PR_out  = widgets.Output()
PR_perf = widgets.HTML()

def update_PR(change):
    PR_out.clear_output(wait=True)
    PR = PR_slider.value
    Tt, Pt, Fn, TSFC, OPR = run_design(PR=PR)
    PR_perf.value = (
        f'<div style="font-family:monospace;color:#f0f6fc;background:#161b22;'
        f'padding:8px 16px;border-radius:6px;display:inline-block;margin:6px 0;">'
        f'Fn = <b style="color:#06b6d4">{Fn:.1f} lbf</b> &nbsp;|&nbsp; '
        f'TSFC = <b style="color:#f59e0b">{TSFC:.5f}</b> &nbsp;|&nbsp; '
        f'OPR = <b style="color:#a78bfa">{OPR:.3f}</b>'
        f'</div>'
    )
    with PR_out:
        fig1 = make_station_fig(Tt, Pt, Tt_ref, Pt_ref, 'PR', f'{PR:.1f}')
        display(widgets.HTML(f'<img src="{fig_to_img(fig1)}" style="max-width:100%">'))
        fig2 = make_ts_fig(Tt, Pt, Tt_ref, Pt_ref, 'PR', f'{PR:.1f}')
        display(widgets.HTML(f'<img src="{fig_to_img(fig2)}" style="max-width:100%">'))

PR_slider.observe(update_PR, names='value')

q3 = reveal_button(
    "As you increase PR from 8 to 20, TSFC drops significantly. What physical principle drives this, "
    "and what is the design limit that prevents you from just using PR=100?",
    "Higher PR increases the <b>thermal efficiency</b> of the Brayton cycle — the same amount of "
    "heat added does more work. The limit is the <b>compressor exit temperature (Tt2)</b>: "
    "at very high PR, Tt2 approaches T4, leaving no temperature margin for combustion. "
    "You'd need to add more fuel just to maintain T4, erasing the efficiency gain."
)
q4 = reveal_button(
    "Look at the T–s diagram as you change PR. What happens to the 'width' of the cycle "
    "(the entropy change from inlet to nozzle) as PR increases?",
    "The cycle width <b>narrows</b> — a more efficient cycle produces the same work with less "
    "irreversibility (entropy generation). The compressor and turbine legs become steeper "
    "on the T–s diagram, which is the signature of a higher-efficiency thermodynamic cycle."
)

display(widgets.VBox([PR_slider, PR_perf, q3, q4, PR_out]))
update_PR(None)


---
## 📉 Section 3 — Compressor Efficiency (η_c)

Compressor efficiency measures how close the real compression process is to ideal (isentropic).
A lower η_c means the compressor wastes more shaft work as heat, raising exit temperature for the same PR.
Typical values: 0.78–0.88 for modern axial compressors.


In [ ]:
etac_slider = widgets.FloatSlider(
    value=0.83, min=0.70, max=0.92, step=0.01,
    description='η_c:', style={'description_width': '80px'},
    layout=widgets.Layout(width='500px'),
    continuous_update=False,
    readout_format='.2f'
)
etac_out  = widgets.Output()
etac_perf = widgets.HTML()

def update_etac(change):
    etac_out.clear_output(wait=True)
    eta_c = etac_slider.value
    Tt, Pt, Fn, TSFC, OPR = run_design(eta_c=eta_c)
    etac_perf.value = (
        f'<div style="font-family:monospace;color:#f0f6fc;background:#161b22;'
        f'padding:8px 16px;border-radius:6px;display:inline-block;margin:6px 0;">'
        f'Fn = <b style="color:#06b6d4">{Fn:.1f} lbf</b> &nbsp;|&nbsp; '
        f'TSFC = <b style="color:#f59e0b">{TSFC:.5f}</b> &nbsp;|&nbsp; '
        f'Tt_comp_exit = <b style="color:#f43f5e">{Tt[2]:.1f}°R</b>'
        f'</div>'
    )
    with etac_out:
        fig1 = make_station_fig(Tt, Pt, Tt_ref, Pt_ref, 'eta_c', f'{eta_c:.2f}')
        display(widgets.HTML(f'<img src="{fig_to_img(fig1)}" style="max-width:100%">'))
        fig2 = make_ts_fig(Tt, Pt, Tt_ref, Pt_ref, 'eta_c', f'{eta_c:.2f}')
        display(widgets.HTML(f'<img src="{fig_to_img(fig2)}" style="max-width:100%">'))

etac_slider.observe(update_etac, names='value')

q5 = reveal_button(
    "If compressor efficiency drops from 0.83 to 0.75 (e.g. due to compressor blade erosion), "
    "what happens to the compressor exit temperature Tt2, and why does that hurt the burner?",
    "Tt2 <b>rises</b> — a less efficient compressor wastes shaft work as heat, so more of the "
    "compression work ends up as temperature rather than pressure. This is bad for the burner "
    "because the <b>available temperature rise (T4 − Tt2) shrinks</b>, so you need more FAR "
    "(fuel-to-air ratio) to hit the same T4, which drives TSFC up."
)
q6 = reveal_button(
    "On the T–s diagram, what does a lower compressor efficiency look like geometrically, "
    "and what does it imply about entropy generation?",
    "The compressor leg <b>tilts rightward</b> — entropy increases during compression because "
    "the process is no longer isentropic. A perfectly efficient compressor would be a vertical "
    "line on the T–s diagram. The rightward tilt is exactly the irreversibility introduced by "
    "friction and flow separation in the compressor blades."
)

display(widgets.VBox([etac_slider, etac_perf, q5, q6, etac_out]))
update_etac(None)


---
## ⚡ Section 4 — Turbine Efficiency (η_t)

Turbine efficiency measures how well the turbine converts gas enthalpy into shaft work.
Lower η_t means less work extracted per unit temperature drop, so the turbine exit is hotter.
Typical values: 0.84–0.92.


In [ ]:
etat_slider = widgets.FloatSlider(
    value=0.86, min=0.72, max=0.94, step=0.01,
    description='η_t:', style={'description_width': '80px'},
    layout=widgets.Layout(width='500px'),
    continuous_update=False,
    readout_format='.2f'
)
etat_out  = widgets.Output()
etat_perf = widgets.HTML()

def update_etat(change):
    etat_out.clear_output(wait=True)
    eta_t = etat_slider.value
    Tt, Pt, Fn, TSFC, OPR = run_design(eta_t=eta_t)
    etat_perf.value = (
        f'<div style="font-family:monospace;color:#f0f6fc;background:#161b22;'
        f'padding:8px 16px;border-radius:6px;display:inline-block;margin:6px 0;">'
        f'Fn = <b style="color:#06b6d4">{Fn:.1f} lbf</b> &nbsp;|&nbsp; '
        f'TSFC = <b style="color:#f59e0b">{TSFC:.5f}</b> &nbsp;|&nbsp; '
        f'Tt_turb_exit = <b style="color:#10b981">{Tt[4]:.1f}°R</b>'
        f'</div>'
    )
    with etat_out:
        fig1 = make_station_fig(Tt, Pt, Tt_ref, Pt_ref, 'eta_t', f'{eta_t:.2f}')
        display(widgets.HTML(f'<img src="{fig_to_img(fig1)}" style="max-width:100%">'))
        fig2 = make_ts_fig(Tt, Pt, Tt_ref, Pt_ref, 'eta_t', f'{eta_t:.2f}')
        display(widgets.HTML(f'<img src="{fig_to_img(fig2)}" style="max-width:100%">'))

etat_slider.observe(update_etat, names='value')

q7 = reveal_button(
    "If turbine efficiency drops, the turbine exit temperature (Tt4) rises. "
    "Why does that also increase the nozzle exit velocity — and does that help or hurt thrust?",
    "A higher turbine exit temperature means <b>more enthalpy is left in the gas</b> when it "
    "reaches the nozzle. The nozzle converts this to kinetic energy, so exit velocity increases. "
    "In the short term this can <b>partially compensate for thrust</b>, but the shaft power "
    "delivered to the compressor is lower (turbine extracted less work), so the compressor "
    "eventually stalls unless the balance re-adjusts — which is why TSFC still rises."
)
q8 = reveal_button(
    "The turbine must extract exactly enough work to drive the compressor (shaft power balance). "
    "If η_t drops, what must happen to the turbine pressure ratio to maintain shaft balance?",
    "The turbine pressure ratio must <b>increase</b> — the turbine must expand the gas across a "
    "larger pressure drop to extract the same shaft work, because each unit of pressure drop now "
    "yields less work (lower efficiency). This is why Pt at the turbine exit drops more steeply "
    "as η_t decreases."
)

display(widgets.VBox([etat_slider, etat_perf, q7, q8, etat_out]))
update_etat(None)
